# ReAct (Reasoning + Acting) Agent — AWS Bedrock

## What is ReAct?

ReAct = **Re**asoning + **Act**ing

The agent thinks, calls a real tool, looks at the result, then thinks again — until it has the final answer.

```
Thought     → What do I need to do?
Action      → Call a tool (calculator, price lookup, weather...)
Observation → See the REAL result from the tool
Thought     → What does that mean? Do I need another action?
... repeat ...
Final Answer
```

---
## Step 1: Setup — Bedrock + Tools

- `get_completion()` now accepts `stop_sequences` — this is the key fix for the hallucination bug
- All tools strip quotes from input — this is the fix for the `'Tokyo'` vs `Tokyo` bug
- Tools return plain numbers (no `$` sign) so the calculator gets clean input like `1099 * 0.15`

In [ ]:
import boto3
import json
import re

MODEL_NAME = "us.amazon.nova-lite-v1:0"
# MODEL_NAME = "anthropic.claude-3-haiku-20240307-v1:0"
AWS_REGION = "us-east-1"

bedrock = boto3.client('bedrock-runtime', region_name=AWS_REGION)


# ── Helper: clean quotes the model might add to tool inputs ───────────────
def clean_input(text):
    """Strip quotes and whitespace from tool input.
    Fixes: get_weather('Tokyo') or get_weather(\"Tokyo\") → get_weather(Tokyo)
    """
    return text.strip().strip('"').strip("'")


# ── Bedrock call with stop_sequences support ───────────────────────────────
def get_completion(prompt, system='', stop_sequences=None, model_name=MODEL_NAME):
    """
    Send a prompt to Bedrock and return the response.
    stop_sequences: model stops generating when it hits one of these strings.
    Key fix: stops model from writing fake Observations on its own.
    """
    if model_name.startswith("anthropic.claude"):
        body_dict = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1024,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "top_p": 1,
            "system": system
        }
        if stop_sequences:
            body_dict["stop_sequences"] = stop_sequences
        response = bedrock.invoke_model(body=json.dumps(body_dict), modelId=model_name)
        response_body = json.loads(response.get("body").read())
        return response_body.get("content")[0].get("text")

    elif "amazon.nova" in model_name:
        body_dict = {
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {
                "max_new_tokens": 1024,
                "temperature": 0.0,
                "top_p": 1,
            }
        }
        if system:
            body_dict["system"] = [{"text": system}]
        if stop_sequences:
            body_dict["inferenceConfig"]["stopSequences"] = stop_sequences
        response = bedrock.invoke_model(body=json.dumps(body_dict), modelId=model_name)
        response_body = json.loads(response.get("body").read())
        return response_body.get("output").get("message").get("content")[0].get("text")

    else:
        raise ValueError(f"Unsupported model: {model_name}")


# ── Tool definitions ───────────────────────────────────────────────────────
def calculator(expression):
    """Evaluates a math expression. Returns the result as a string."""
    expression = clean_input(expression)
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(round(result, 2))
    except Exception as e:
        return f"Error evaluating expression: {e}"


def get_product_price(product):
    """Returns the price of a product as a plain number (no $ sign)."""
    product = clean_input(product)  # fix: strips quotes like 'MacBook Air'
    prices = {
        "iphone 15": 799,
        "macbook air": 1099,
        "airpods": 179,
        "ipad": 449,
    }
    for key, price in prices.items():
        if key in product.lower():
            return str(price)  # plain number so calculator works cleanly
    return "Product not found"


def get_weather(city):
    """Returns weather for a city."""
    city = clean_input(city)  # fix: strips quotes like 'Tokyo' or \"Tokyo\"
    weather = {
        "london": "12C, cloudy",
        "new york": "22C, sunny",
        "tokyo": "18C, partly cloudy",
        "chennai": "34C, hot and humid",
        "bangalore": "24C, pleasant",
        "mumbai": "30C, humid",
    }
    return weather.get(city.lower(), f"No weather data for '{city}'")


TOOLS = {
    "calculator": calculator,
    "get_product_price": get_product_price,
    "get_weather": get_weather,
}

print("Setup complete!")
print(f"Tools available: {list(TOOLS.keys())}")

# Quick sanity check on the fix
print("\nTool sanity checks:")
print(f"  get_weather('Tokyo') with quotes  → {get_weather(chr(39)+'Tokyo'+chr(39))}")
print(f"  get_weather(Tokyo) without quotes → {get_weather('Tokyo')}")
print(f"  get_product_price(MacBook Air)    → {get_product_price('MacBook Air')}")
print(f"  calculator(1099 * 0.15)           → {calculator('1099 * 0.15')}")

---
## Step 2: System Prompt + ReAct Agent

The system prompt tells the model:
- Use exactly one Thought + one Action per response, then **stop**
- Never write `Observation:` yourself — wait for it
- Pass tool inputs **without quotes** so the tools receive clean values

The agent loop:
1. Calls the model with `stop_sequences=["Observation:"]`
2. Parses the `Action:` line
3. Strips any quotes from the input using `clean_input()`
4. Calls the real tool
5. Appends the real `Observation:` to the conversation
6. Repeats until `Final Answer:` is found

In [ ]:
REACT_SYSTEM_PROMPT = """You are a problem-solving assistant. Solve problems step by step using tools.

Available tools:
- calculator(expression) : evaluates math. Always use plain numbers, no $ signs.
  Example: calculator(1099 * 0.15)
- get_product_price(product) : returns price as a plain number.
  Example: get_product_price(MacBook Air)
- get_weather(city) : returns current weather for a city.
  Example: get_weather(Tokyo)

STRICT RULES:
1. Write exactly ONE Thought and ONE Action per response, then stop immediately.
2. Never write Observation yourself. It will be provided to you.
3. Never write multiple steps ahead or simulate future steps.
4. Pass tool inputs WITHOUT quotes. Write get_weather(Tokyo) not get_weather('Tokyo').
5. When you have enough information, write: Final Answer: <your answer>

Format:
Thought: <your reasoning about what to do next>
Action: <tool_name>(<input>)"""


def run_react_agent(question, max_steps=8):

    print(f"QUESTION: {question}")
    print("=" * 60)

    conversation = f"Question: {question}"

    for step in range(1, max_steps + 1):
        print(f"\n--- Step {step} ---")

        # FIX 1: stop_sequences stops the model before it writes fake Observations
        response = get_completion(
            conversation,
            system=REACT_SYSTEM_PROMPT,
            stop_sequences=["\nObservation:", "Observation:"]
        )
        print(response.strip())

        # Check if we are done
        if "Final Answer:" in response:
            final = response.split("Final Answer:")[-1].strip()
            print(f"\n{'=' * 60}")
            print(f"FINAL ANSWER: {final}")
            return final

        # Parse Action — regex handles both tool(input) and tool('input') and tool(\"input\")
        action_match = re.search(r'Action:\s*(\w+)\([\"\']?(.+?)[\"\']?\)', response)
        if action_match:
            tool_name = action_match.group(1).strip()
            
            tool_input = clean_input(action_match.group(2))

            print(f"  → Calling: {tool_name}({tool_input})")

            if tool_name in TOOLS:
                observation = TOOLS[tool_name](tool_input)
            else:
                observation = f"Unknown tool '{tool_name}'. Available: {list(TOOLS.keys())}"

            print(f"  → Observation: {observation}")

            # Append real observation to conversation history
            conversation += f"\n\n{response.strip()}\nObservation: {observation}"
        else:
            print("No valid Action found in response — stopping.")
            break

    print("Max steps reached without final answer.")
    return None

---
## Step 3: Run All Test Cases

These are the same questions that failed before. All should work correctly now.

In [ ]:
# Test 1: Needs price lookup + calculator (two tool calls)
run_react_agent("What is 15% of the price of a MacBook Air?")

In [ ]:
# Test 2: Weather lookup — was returning 'not available' due to quoted input
run_react_agent("What is the weather in Tokyo?")

In [ ]:
# Test 3: Price lookup + multiplication
run_react_agent("If I buy 3 AirPods, what is the total cost?")

In [ ]:
# Test 4: Two price lookups + subtraction
run_react_agent("What is the price difference between a MacBook Air and an iPad?")